# Staged RL Training for Qwen2.5-VL

This notebook copies the uploaded code dataset into `/kaggle/working`, installs dependencies, and runs one explicit training phase.

In [ ]:
import os, shutil, subprocess, glob, json, tarfile, sys
from datetime import datetime

WORKDIR = "/kaggle/working/RL_GSPO_Qwen2.5VLM"
VENV_DIR = "/tmp/rl-gspo-venv"
VENV_PYTHON = f"{VENV_DIR}/bin/python"
RUN_ID = datetime.utcnow().strftime("%Y%m%dT%H%M%SZ")


def write_progress(message):
    timestamp = datetime.utcnow().isoformat()
    line = f"{timestamp} {message}"
    print(line, flush=True)


write_progress(f"Notebook start (run_id={RUN_ID})")

matches = []
for root, _, files in os.walk("/kaggle/input"):
    if "rl_gspo_qwen2_5vlm_test3.py" in files:
        matches.append(root)
if not matches:
    raise FileNotFoundError(f"Could not find attached code dataset. Visible inputs: {glob.glob('/kaggle/input/*')}")

if len(matches) > 1:
    raise RuntimeError(f"Ambiguous code dataset attachment. Matches={matches}. Visible inputs: {glob.glob('/kaggle/input/*')}")

src = matches[0]
if os.path.exists(WORKDIR):
    shutil.rmtree(WORKDIR)
shutil.copytree(src, WORKDIR)
for archive_name in ("staged_rl.tar", "tests.tar"):
    archive_path = os.path.join(WORKDIR, archive_name)
    if os.path.exists(archive_path):
        with tarfile.open(archive_path, "r") as tf:
            tf.extractall(WORKDIR)
        os.remove(archive_path)
        print("Extracted", archive_name)
for folder_name in ("staged_rl", "tests"):
    nested_path = os.path.join(WORKDIR, folder_name, folder_name)
    target_path = os.path.join(WORKDIR, folder_name)
    if os.path.isdir(nested_path):
        temp_path = f"{target_path}_flat"
        if os.path.exists(temp_path):
            shutil.rmtree(temp_path)
        shutil.move(nested_path, temp_path)
        shutil.rmtree(target_path)
        shutil.move(temp_path, target_path)
        print("Flattened", folder_name)
print("Using code dataset", src)
print("Copied code to", WORKDIR)
write_progress("Copied code dataset")
print(sorted(os.listdir(WORKDIR)))


In [ ]:
write_progress("Installing uv and creating venv")
subprocess.run(["python3", "-m", "pip", "install", "-q", "uv"], check=True)
subprocess.run(["python3", "-m", "uv", "venv", "--seed", "--clear", VENV_DIR], check=True)
install_commands = [
    [VENV_PYTHON, "-m", "pip", "install", "--no-cache-dir", "--upgrade", "pip", "setuptools", "wheel"],
    [
        VENV_PYTHON,
        "-m",
        "pip",
        "install",
        "--no-cache-dir",
        "numpy==1.26.4",
        "scipy==1.15.3",
        "scikit-learn==1.6.1",
    ],
    [
        VENV_PYTHON,
        "-m",
        "pip",
        "install",
        "--no-cache-dir",
        "pillow",
        "packaging",
        "safetensors",
        "torchvision",
        "bitsandbytes",
        "xformers",
        "huggingface_hub>=0.34.0",
        "datasets==4.3.0",
        "transformers==4.56.2",
        "trl==0.22.2",
        "unsloth",
        "modelscope",
    ],
    [
        VENV_PYTHON,
        "-m",
        "pip",
        "install",
        "--no-cache-dir",
        "vllm==0.10.2",
        "--extra-index-url",
        "https://wheels.vllm.ai/0.10.2/",
    ],
]
for command in install_commands:
    print("Running:", " ".join(command))
    subprocess.run(command, check=True, cwd=WORKDIR)
write_progress("Finished venv package installs")
print("Venv ready:", VENV_PYTHON)


In [ ]:
compat_script = """
import numpy
import scipy
import sklearn
import vllm
from vllm.sampling_params import GuidedDecodingParams
from trl import GRPOTrainer
print('numpy version:', numpy.__version__)
print('scipy version:', scipy.__version__)
print('sklearn version:', sklearn.__version__)
print('vllm version:', vllm.__version__)
print('compat imports OK')
"""
subprocess.run([VENV_PYTHON, "-c", compat_script], check=True, cwd=WORKDIR)
write_progress("Verified dependencies in venv")

In [ ]:
write_progress("Progress logging via stdout only")


In [ ]:
HARDWARE_PROFILE = "kaggle_t4"
TRAIN_SPLIT = "test"
EVAL_SPLIT = "testmini"
MAX_EVAL_EXAMPLES_PER_SUBSET = 4
def resolve_model_path():
    candidates = glob.glob("/kaggle/input/**/config.json", recursive=True)
    matches = []
    for config_path in candidates:
        root = os.path.dirname(config_path)
        if os.path.exists(os.path.join(root, "model.safetensors")):
            matches.append(root)
    if not matches:
        raise FileNotFoundError(
            f"Could not find model config.json + model.safetensors under /kaggle/input."
        )
    if len(matches) > 1:
        write_progress(f"Multiple model candidates found: {matches}")
    return matches[0]

MODEL_PATH = resolve_model_path()
PHASE_RUNS = [
    ("phase_a", None),
    ("phase_b", "best_structure"),
    ("phase_c", "best_composite"),
]

env = dict(os.environ)
env["PYTHONUNBUFFERED"] = "1"
env["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
env["UNSLOTH_USE_MODELSCOPE"] = "0"
env["TRANSFORMERS_OFFLINE"] = "0"
env["HF_HUB_OFFLINE"] = "0"
env["HF_DATASETS_OFFLINE"] = "0"

if not os.path.exists(os.path.join(MODEL_PATH, "config.json")):
    raise FileNotFoundError(
        f"Expected config.json under {MODEL_PATH}. Check the attached model dataset path."
    )
write_progress(f"Resolved model path: {MODEL_PATH}")

for PHASE, RESUME in PHASE_RUNS:
    cmd = [
        VENV_PYTHON,
        "rl_gspo_qwen2_5vlm_test3.py",
        "--phase", PHASE,
        "--hardware-profile", HARDWARE_PROFILE,
        "--train-split", TRAIN_SPLIT,
        "--eval-split", EVAL_SPLIT,
        "--base-model-path", MODEL_PATH,
        "--max-eval-examples-per-subset", str(MAX_EVAL_EXAMPLES_PER_SUBSET),
    ]
    if RESUME:
        cmd.extend(["--resume", RESUME])

    phase_output_dir = os.path.join(WORKDIR, "outputs_staged", PHASE)
    os.makedirs(phase_output_dir, exist_ok=True)
    train_log_path = os.path.join(phase_output_dir, "train_subprocess.log")
    write_progress(f"Starting phase {PHASE} (resume={RESUME or 'none'})")
    print("Running:", " ".join(cmd))
    print("Subprocess log:", train_log_path)
    with open(train_log_path, "w", encoding="utf-8") as log_file:
        process = subprocess.Popen(
            cmd,
            cwd=WORKDIR,
            env=env,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        for line in process.stdout:
            log_file.write(line)
            log_file.flush()
            print(line, end="", flush=True)
        return_code = process.wait()
    if return_code != 0:
        write_progress(f"Phase {PHASE} failed with return code {return_code}")
        print(f"Phase {PHASE} failed with return code {return_code}. Last 200 log lines:")
        with open(train_log_path, "r", encoding="utf-8") as log_file:
            tail_lines = log_file.readlines()[-200:]
        print("".join(tail_lines))
        raise subprocess.CalledProcessError(return_code, cmd)
    write_progress(f"Phase {PHASE} completed successfully")
    print(f"Phase {PHASE} completed successfully.")


In [ ]:
outputs_root = os.path.join(WORKDIR, "outputs_staged")
collected = []
for root, _, files in os.walk(outputs_root):
    for file_name in files:
        collected.append(os.path.join(root, file_name))
for path in sorted(collected)[-50:]:
    print(path)